In [0]:
import logging
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO)

try:
    logging.info("Gold started")

    df = spark.table("fraud_catalog.silver.transactions_clean")

    logging.info(f"Rows read: {df.count()}")

except Exception as e:
    logging.error(str(e))
    raise

INFO:root:Gold started
INFO:root:Rows read: 1048575


In [0]:
# rule: multiple transactions in one day

window_day = Window.partitionBy("cc_num", "txn_date")

df = df.withColumn("txn_count_per_day", F.count("*").over(window_day)) \
       .withColumn("rule_multi_txn_day",F.when(F.col("txn_count_per_day") > 5, 1).otherwise(0))

logging.info("Rule applied: multiple transactions per day")

INFO:root:Rule applied: multiple transactions per day


In [0]:
display(df.select("cc_num", "txn_date", "txn_count_per_day", "rule_multi_txn_day").limit(10))

cc_num,txn_date,txn_count_per_day,rule_multi_txn_day
1.80011E+14,2012-01-05,5,0
1.80011E+14,2012-01-05,5,0
1.80011E+14,2012-01-05,5,0
1.80011E+14,2012-01-05,5,0
1.80011E+14,2012-01-05,5,0
1.80011E+14,2012-01-10,1,0
1.80011E+14,2012-01-12,4,0
1.80011E+14,2012-01-12,4,0
1.80011E+14,2012-01-12,4,0
1.80011E+14,2012-01-12,4,0


In [0]:
# rule: merchant is too far from customer location

df = df.withColumn("distance_km",6371 * 2 * F.asin( F.sqrt(
            F.pow(F.sin((F.radians(F.col("merch_lat")) - F.radians(F.col("cust_lat"))) / 2), 2) +
            F.cos(F.radians(F.col("cust_lat"))) *
            F.cos(F.radians(F.col("merch_lat"))) *
            F.pow(F.sin((F.radians(F.col("merch_long")) - F.radians(F.col("cust_long"))) / 2), 2))))

df = df.withColumn("rule_distance_anomaly",F.when(F.col("distance_km") > 300, 1).otherwise(0))

logging.info("Rule applied: distance anomaly")

INFO:root:Rule applied: distance anomaly


In [0]:
display(df.select("cust_lat", "cust_long", "merch_lat", "merch_long", "distance_km", "rule_distance_anomaly").limit(10))

cust_lat,cust_long,merch_lat,merch_long,distance_km,rule_distance_anomaly
30.8985,-87.1332,31.414174,-87.138632,57.34266223290291,0
37.3304,-121.7913,37.548452,-121.536454,33.07759438519743,0
36.2395,-95.9596,35.891929,-96.473787,60.246689175562246,0
42.1767,-79.9416,43.011965,-80.629362,108.60708358490243,0
42.4969,-83.2911,41.920889,-83.296441,64.05101144666448,0
41.8948,-73.9767,42.285479,-73.130929,82.20551515160696,0
42.0765,-87.7246,42.727061,-86.930984,97.36062159987242,0
31.9571,-98.9656,31.432831,-98.600583,67.75729756754166,0
33.9659,-80.9355,34.808248,-81.810611,123.37449427217808,0
39.8839,-75.4669,40.781172,-76.249659,119.81840781517678,0


In [0]:
# rule: transactions happening too quickly one after another

time_window = Window.partitionBy("cc_num").orderBy("txn_ts")

df = df.withColumn("prev_txn_ts", F.lag("txn_ts").over(time_window))

df = df.withColumn("time_diff_sec",F.col("txn_ts").cast("long") - F.col("prev_txn_ts").cast("long"))

df = df.withColumn("rule_rapid_txn",F.when(F.col("prev_txn_ts").isNotNull() & (F.col("time_diff_sec") <= 300),1).otherwise(0))

logging.info("Rule applied: rapid transactions")

display(df.select("cc_num", "txn_ts", "prev_txn_ts", "time_diff_sec", "rule_rapid_txn").limit(10))

INFO:root:Rule applied: rapid transactions


cc_num,txn_ts,prev_txn_ts,time_diff_sec,rule_rapid_txn
1.80014E+14,2012-01-01T01:23:26Z,null,null,0
1.80014E+14,2012-01-01T16:26:09Z,2012-01-01T01:23:26Z,54163,0
1.80014E+14,2012-01-01T18:51:06Z,2012-01-01T16:26:09Z,8697,0
1.80014E+14,2012-01-02T19:12:12Z,2012-01-01T18:51:06Z,87666,0
1.80014E+14,2012-01-04T14:32:24Z,2012-01-02T19:12:12Z,156012,0
1.80014E+14,2012-01-04T23:55:14Z,2012-01-04T14:32:24Z,33770,0
1.80014E+14,2012-01-05T03:01:24Z,2012-01-04T23:55:14Z,11170,0
1.80014E+14,2012-01-06T01:58:12Z,2012-01-05T03:01:24Z,82608,0
1.80014E+14,2012-01-06T22:50:15Z,2012-01-06T01:58:12Z,75123,0
1.80014E+14,2012-01-07T09:29:34Z,2012-01-06T22:50:15Z,38359,0


In [0]:
# create fraud score

df = df.withColumn("fraud_score",
    F.col("is_high_value") * 30 +
    F.col("is_night_txn") * 10 +
    F.col("rule_multi_txn_day") * 20 +
    F.col("rule_distance_anomaly") * 25 +
    F.col("rule_rapid_txn") * 15)

# create final fraud flag

df = df.withColumn("fraud_flag",F.when(F.col("fraud_score") >= 40, 1).otherwise(0))

logging.info("Fraud score and fraud flag created")

INFO:root:Fraud score and fraud flag created


In [0]:
display(df.select("amt", "is_high_value", "is_night_txn", "rule_multi_txn_day", "rule_distance_anomaly", "rule_rapid_txn", "fraud_score", "fraud_flag").limit(10))

amt,is_high_value,is_night_txn,rule_multi_txn_day,rule_distance_anomaly,rule_rapid_txn,fraud_score,fraud_flag
5.83,0,1,0,0,0,10,0
47.58,0,0,0,0,0,0,0
77.7,0,0,0,0,0,0,0
83.51,0,0,0,0,0,0,0
1.41,0,0,0,0,0,0,0
83.1,0,1,0,0,0,10,0
106.77,0,1,0,0,0,10,0
20.05,0,1,0,0,0,10,0
49.03,0,1,0,0,0,10,0
78.03,0,0,0,0,0,0,0


In [0]:
# risk level
df = df.withColumn("risk_level",F.when(F.col("fraud_score") >= 70, "HIGH")
     .when(F.col("fraud_score") >= 40, "MEDIUM").otherwise("LOW"))

# fraud reason
df = df.withColumn("fraud_reason",F.concat_ws(", ",
        F.when(F.col("is_high_value") == 1, "High Amount"),
        F.when(F.col("is_night_txn") == 1, "Night Transaction"),
        F.when(F.col("rule_multi_txn_day") == 1, "Multiple Txn"),
        F.when(F.col("rule_distance_anomaly") == 1, "Location Mismatch"),
        F.when(F.col("rule_rapid_txn") == 1, "Rapid Txn")))

df = df.withColumn("fraud_reason",F.when(F.col("fraud_reason") == "", "Normal").otherwise(F.col("fraud_reason")))

logging.info("Risk level and fraud reason created")

INFO:root:Risk level and fraud reason created


In [0]:
df = df.withColumn("alert_priority",F.when(F.col("fraud_score") >= 70, "P1").when(F.col("fraud_score") >= 40, "P2").otherwise("P3"))

In [0]:
df = df.withColumn("rule_hit_count",
    F.col("is_high_value") +
    F.col("is_night_txn") +
    F.col("rule_multi_txn_day") +
    F.col("rule_distance_anomaly") +
    F.col("rule_rapid_txn"))

In [0]:
df = df.withColumn("investigation_status",F.when(F.col("fraud_flag") == 1, "Open").otherwise("Not Required"))

In [0]:
# save final gold table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_transactions") \
  .partitionBy("txn_date") \
  .saveAsTable("fraud_catalog.gold.fraud_transactions")

logging.info("Gold table saved successfully")

INFO:root:Gold table saved successfully


In [0]:
spark.sql("""
select fraud_flag, risk_level, count(*) as record_count
from fraud_catalog.gold.fraud_transactions
group by fraud_flag, risk_level
order by fraud_flag, risk_level
""").show()

+----------+----------+------------+
|fraud_flag|risk_level|record_count|
+----------+----------+------------+
|         0|       LOW|     1043372|
|         1|      HIGH|          45|
|         1|    MEDIUM|        5158|
+----------+----------+------------+

